# 06 -- Stage 6: Final Export & ONNX

Saves the fully-trained `models/final_model.pth`, exports it to
`models/final_model.onnx` via `services/detection/export_onnx.py`,
and verifies the ONNX output matches the PyTorch model's output.

**After this step**: copy `models/final_model.onnx` to
`models/image_model.onnx` (or update `config/settings.yaml`'s
`models.image_model_onnx_path`) to make the live app use it -- no
serving code changes required.

In [ ]:
import sys, os
PROJECT_ROOT = "/content/ForensicAI_Lab"  # adjust if you cloned/uploaded elsewhere
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # fallback: running locally from the project root
sys.path.insert(0, PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
else:
    print("Not in Colab -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import torch
from config.loader import get_settings, resolve_path

settings = get_settings()

# At this point `model` should be your fully fine-tuned model from Stage 5.
# If running this notebook standalone, load your best checkpoint instead:
# from services.detection.model import ForensicMultiHeadNet
# model = ForensicMultiHeadNet(backbone=settings.models.backbone)
# model.load_state_dict(torch.load(resolve_path(settings.models.image_model_checkpoint)))

final_checkpoint_path = resolve_path(settings.models.final_model_checkpoint)
final_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), final_checkpoint_path)
print("Saved final checkpoint to", final_checkpoint_path)

In [ ]:
from services.detection.export_onnx import export_to_onnx

onnx_path = export_to_onnx(
    output_path=settings.models.final_model_onnx_path,
    checkpoint_path=settings.models.final_model_checkpoint,
)
print("Exported to", onnx_path)

In [ ]:
# Verify the ONNX export matches the PyTorch model's output on a random input.
import numpy as np
import onnxruntime as ort

model.eval()
dummy = torch.randn(1, settings.inference.input_channels, *settings.inference.image_size)
with torch.no_grad():
    torch_out = model(dummy)

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_out = session.run(
    settings.inference.output_names,
    {settings.inference.input_name: dummy.numpy()},
)

for name, onnx_arr in zip(settings.inference.output_names, onnx_out):
    torch_arr = torch_out[name].detach().numpy()
    close = np.allclose(torch_arr, onnx_arr, atol=1e-4)
    print(f"{name}: allclose={close}, max_abs_diff={np.abs(torch_arr - onnx_arr).max():.6f}")